# 02 — Resposta às perguntas de investigação

Este notebook utiliza as variáveis criadas por `src/features/build_features.py` para estudar três perguntas:

1. Como evoluíram os preços da habitação em cada município entre 2021 e 2025?
2. Que relação existe entre rendimento fiscal e preço da habitação entre 2021 e 2024?
3. O saldo migratório por mil habitantes está associado ao crescimento anual dos preços?

A análise é observacional e exploratória. Associações estatísticas não demonstram causalidade.


## 1. Preparação

O notebook lê a tabela enriquecida. Se o ficheiro ainda não existir, executa primeiro `python -m src.features.build_features` na raiz do projeto.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Não foi possível encontrar a raiz do projeto.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
FEATURE_PATH = PROJECT_ROOT / "data/processed/municipality_year_features_2021_2025.csv"
if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        "Tabela de features não encontrada. Executa: python -m src.features.build_features"
    )

features = pd.read_csv(FEATURE_PATH, dtype={"municipality_code": "string"})
assert len(features) == 1540
assert not features.duplicated(["municipality_code", "year"]).any()
print(f"Linhas: {len(features):,} | Colunas: {features.shape[1]}")
features.head()


## 2. Pergunta 1 — Como evoluíram os preços entre 2021 e 2025?

Para uma comparação direta usamos apenas municípios com preço disponível nos dois extremos. A variação acumulada é:

\[
\left(rac{preço_{2025}}{preço_{2021}}-1ight)	imes 100
\]


In [ ]:
price_wide = features.pivot(
    index=["municipality_code", "municipality_name"],
    columns="year",
    values="housing_price_m2_total",
)

price_change = price_wide[[2021, 2025]].dropna().copy()
price_change["growth_2021_2025_pct"] = (
    price_change[2025].div(price_change[2021]).sub(1).mul(100)
)
price_change["annualised_growth_pct"] = (
    price_change[2025].div(price_change[2021]).pow(1 / 4).sub(1).mul(100)
)

print(f"Municípios comparáveis: {len(price_change)} de 308")
print(f"Com aumento: {(price_change['growth_2021_2025_pct'] > 0).sum()}")
print(f"Com redução: {(price_change['growth_2021_2025_pct'] < 0).sum()}")
price_change[["growth_2021_2025_pct", "annualised_growth_pct"]].describe()


In [ ]:
ax = price_change["growth_2021_2025_pct"].plot(
    kind="hist",
    bins=25,
    figsize=(10, 5),
    edgecolor="white",
    title="Variação municipal do preço entre 2021 e 2025",
)
ax.axvline(0, color="black", linewidth=1)
ax.axvline(
    price_change["growth_2021_2025_pct"].median(),
    color="darkred",
    linestyle="--",
    label="Mediana",
)
ax.set_xlabel("Variação acumulada (%)")
ax.set_ylabel("Número de municípios")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
top_growth = price_change.nlargest(10, "growth_2021_2025_pct").reset_index()
bottom_growth = price_change.nsmallest(10, "growth_2021_2025_pct").reset_index()

print("Maiores aumentos:")
display(top_growth[["municipality_name", 2021, 2025, "growth_2021_2025_pct"]])
print("Menores variações:")
display(bottom_growth[["municipality_name", 2021, 2025, "growth_2021_2025_pct"]])


### Conclusão da pergunta 1

Foi possível comparar 298 municípios. Destes, **291 registaram aumento** e sete registaram redução. A variação mediana foi aproximadamente **56,8%**, enquanto a média foi 57,5%.

Fornos de Algodres apresenta a maior variação percentual, de 159 para 464 €/m² (+191,8%). Uma percentagem elevada a partir de um preço inicial baixo deve ser interpretada com cuidado. Os resultados mostram uma subida generalizada, mas muito desigual entre municípios. Os valores são nominais e não estão corrigidos pela inflação.


## 3. Pergunta 2 — Que relação existe entre rendimento e preço?

Calculamos a correlação separadamente em cada ano disponível. Também analisamos a percentagem do rendimento fiscal anual por agregado correspondente ao preço mediano de um metro quadrado. Esta percentagem é apenas uma aproximação comparativa, não uma taxa oficial de esforço.


In [ ]:
price_income_by_year = []
for year, group in features.loc[features["year"].le(2024)].groupby("year"):
    valid = group[[
        "housing_price_m2_total",
        "income_after_tax_per_tax_household",
    ]].dropna()
    price_income_by_year.append(
        {
            "year": year,
            "municipalities": len(valid),
            "correlation": valid.corr().iloc[0, 1],
        }
    )

price_income_by_year = pd.DataFrame(price_income_by_year)
price_income_by_year


In [ ]:
ax = price_income_by_year.plot(
    x="year",
    y="correlation",
    marker="o",
    ylim=(0, 1),
    figsize=(9, 4),
    legend=False,
    title="Correlação municipal entre rendimento e preço",
)
ax.set_xlabel("Ano")
ax.set_ylabel("Correlação de Pearson")
plt.tight_layout()
plt.show()


In [ ]:
proxy_by_year = features.groupby("year")[
    "annual_income_needed_for_one_m2_pct"
].median().dropna()

highest_proxy_2024 = (
    features.loc[features["year"].eq(2024)]
    .nlargest(10, "annual_income_needed_for_one_m2_pct")
    [[
        "municipality_name",
        "annual_income_needed_for_one_m2_pct",
        "housing_price_m2_total",
        "income_after_tax_per_tax_household",
    ]]
    .reset_index(drop=True)
)

print("Mediana municipal do indicador por ano:")
display(proxy_by_year.to_frame("percent"))
print("Valores mais elevados em 2024:")
display(highest_proxy_2024)


### Conclusão da pergunta 2

A correlação preço–rendimento é positiva e relativamente estável: varia entre **0,539 e 0,573** de 2021 a 2024. Municípios com maior rendimento fiscal tendem, portanto, a apresentar preços mais elevados, embora existam muitas diferenças não explicadas pelo rendimento.

A mediana do indicador preço–rendimento permanece próxima de 4,7%–5,0% durante o período. Em 2024, Aljezur, Lagos, Vila do Bispo, Loulé e Albufeira apresentam os valores mais elevados. O indicador compara o preço de apenas 1 m² com o rendimento anual e não mede o custo total de uma habitação.


## 4. Pergunta 3 — Migração e crescimento dos preços

Comparamos o crescimento anual dos preços com o saldo migratório por mil habitantes. Calculamos correlação de Pearson, sensível à magnitude, e de Spearman, baseada na ordenação e menos sensível a valores extremos.


In [ ]:
same_year_results = []
for year, group in features.dropna(
    subset=["housing_price_growth_pct", "migration_rate_per_1000"]
).groupby("year"):
    pair = group[["housing_price_growth_pct", "migration_rate_per_1000"]]
    same_year_results.append(
        {
            "year": year,
            "municipalities": len(pair),
            "pearson": pair.corr(method="pearson").iloc[0, 1],
            "spearman": pair.corr(method="spearman").iloc[0, 1],
        }
    )

same_year_results = pd.DataFrame(same_year_results)
same_year_results


Também testamos se a migração de um ano está associada ao crescimento do preço no ano seguinte. O `shift()` coloca a taxa migratória anterior na linha atual de cada município.


In [ ]:
lagged = features.sort_values(["municipality_code", "year"]).copy()
municipality_groups = lagged.groupby("municipality_code", sort=False)
lagged["migration_rate_lag1"] = municipality_groups["migration_rate_per_1000"].shift()
previous_year = municipality_groups["year"].shift()
lagged.loc[lagged["year"].sub(previous_year).ne(1), "migration_rate_lag1"] = pd.NA

lagged_results = []
for year, group in lagged.dropna(
    subset=["housing_price_growth_pct", "migration_rate_lag1"]
).groupby("year"):
    pair = group[["housing_price_growth_pct", "migration_rate_lag1"]]
    lagged_results.append(
        {
            "year": year,
            "municipalities": len(pair),
            "pearson": pair.corr(method="pearson").iloc[0, 1],
            "spearman": pair.corr(method="spearman").iloc[0, 1],
        }
    )

lagged_results = pd.DataFrame(lagged_results)
lagged_results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, table, title in [
    (axes[0], same_year_results, "Mesmo ano"),
    (axes[1], lagged_results, "Migração do ano anterior"),
]:
    table.plot(x="year", y=["pearson", "spearman"], marker="o", ax=ax)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_ylim(-0.4, 0.4)
    ax.set_title(title)
    ax.set_xlabel("Ano do crescimento do preço")
    ax.set_ylabel("Correlação")

plt.tight_layout()
plt.show()


### Conclusão da pergunta 3

As correlações são fracas e instáveis. Para o mesmo ano, Pearson varia entre aproximadamente -0,030 e 0,190; usando a migração do ano anterior, varia entre -0,126 e 0,162. Spearman apresenta o mesmo padrão geral.

Com estas definições e este período curto, **não encontramos uma associação linear consistente** entre saldo migratório por mil habitantes e crescimento anual dos preços. Este resultado não prova ausência de relação: podem existir atrasos diferentes, efeitos regionais, não linearidade e variáveis omitidas.


## 5. Limitações e próximo passo analítico

Os resultados permitem responder descritivamente às três perguntas, mas ainda existem limitações:

- apenas cinco anos de dados e quatro variações anuais;
- preços e rendimentos nominais, sem correção pela inflação;
- ausência de classificação regional no painel;
- ausência de características da habitação, turismo, oferta e transações;
- correlações simples sem controlo simultâneo de outras variáveis.

O próximo passo é acrescentar uma dimensão regional oficial e avançar para modelos estatísticos de painel com efeitos de ano e região. Antes disso, devemos verificar a robustez dos resultados a valores extremos.
